<a href="https://colab.research.google.com/github/RajendarKumar/ai-agent_phase_1/blob/main/prompt_chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet "groq>=0.4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
# Install pydantic for schema validation
!pip install --quiet pydantic

In [ ]:
from google.colab import userdata
key = userdata.get('GROQ_API_KEY')

In [ ]:
from groq import Groq
import json
import datetime
import re
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, Literal



# Define Pydantic models for output validation
class EmailDetails(BaseModel):
    category: Literal["refund", "shipping", "complaint", "other"]
    order_number: Optional[str]
    amount: float # Assuming amount can be a float, based on 'number' type
    order_date: Optional[str] = Field(pattern="^\\d{4}-\\d{2}-\\d{2}$") # YYYY-MM-DD format

class RefundPolicyResult(BaseModel):
    refund_possible: Literal["YES", "NO", "NOT SURE"]
    reason: str

client = Groq(api_key=key)

def refund_rules_list(refund_policy:str):
  refund_rules_list = []
  # Split the string by rule number (e.g., '1. ' or '2. ')
  rules = re.split(r'\d+\.\s*', refund_policy)

  # Filter out empty strings that might result from the split and process each rule
  for i, rule_text in enumerate(rules):
      if rule_text.strip(): # Ensure the rule text is not empty
          rule_obj = {
              "rule": f"Rule {i}", # Assign a generic rule name, or infer from text if possible
              "value": rule_text.strip()
          }
          refund_rules_list.append(rule_obj)
  print(json.dumps(refund_rules_list, indent=2))
  return refund_rules_list

def extract_email_details(email_body: str, model: str = "openai/gpt-oss-120b"):
    """Extracts structured details from an email body using an LLM and validates against schema."""
    user_prompt = f"""
    Extract the following details from the email body into a JSON object:
    - category (refund, shipping, complaint, other)
    - order_number (string or null)
    - amount (number)
    - order_date (string, format YYYY-MM-DD)

    Email body: "{email_body}"

    JSON Schema:
    ```json
    {{
      "category": {{ "type": "string", "enum": ["refund", "shipping", "complaint", "other"] }},
      "order_number": {{ "type": ["string", "null"] }},
      "amount": {{ "type": "number" }},
      "order_date": {{ "type": ["string", "null"], "format": "YYYY-MM-DD" }}
    }}
    ```
    Ensure the output is valid JSON according to this schema.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=0.0, # Make it deterministic for data extraction
        max_tokens=1024,
        top_p=1.0,
        stream=False,
        response_format={ "type": "json_object" }
    )
    print(f'email evaluation cost : {response.usage}')

    parsed_content = json.loads(response.choices[0].message.content)

    try:
        # Validate the parsed content against the Pydantic model
        validated_data = EmailDetails.model_validate(parsed_content)
        return validated_data.model_dump() # Return as dict
    except ValidationError as e:
        print(f"[ERROR] Email details validation failed: {e}")
        # In a real application, you might log this error and handle it gracefully
        # For now, we'll return the raw parsed content, but a robust system might raise it
        return parsed_content


def apply_refund_policy(extracted_data: dict, refund_policy: list, model: str = "openai/gpt-oss-120b", evaluation_date:datetime.date=None):
    """Applies the refund policy to extracted email data using an LLM and validates output."""
    extracted_data_json = json.dumps(extracted_data, indent=2)

    # Ensure evaluation_date is provided, default to today if not
    if evaluation_date is None:
        evaluation_date = datetime.date.today()
    evaluation_date_str = evaluation_date.strftime('%Y-%m-%d')

    user_prompt = f"""
    Given the following extracted email data, refund policy rules, and the current evaluation date,
    determine if a refund is possible. Provide 'YES', 'NO', or 'NOT SURE' for 'refund_possible'
    and a 'reason' explaining the decision.

    Extracted Email Data:
    ```json
    {extracted_data_json}
    ```

    Refund Policy rules:
    {refund_policy}

    Current Evaluation Date: {evaluation_date_str}

    JSON Schema for output:
    ```json
    {{
      "refund_possible": {{ "type": "string", "enum": ["YES", "NO", "NOT SURE"] }},
      "reason": {{ "type": "string" }}
    }}
    ```
    Ensure the output is valid JSON according to this schema.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=0.0, # Make it deterministic for policy application
        max_tokens=1024,
        top_p=1.0,
        stream=False,
        response_format={ "type": "json_object" }
    )
    print(f'policy evaluation cost : {response.usage}')

    parsed_content = json.loads(response.choices[0].message.content)

    try:
        # Validate the parsed content against the Pydantic model
        validated_data = RefundPolicyResult.model_validate(parsed_content)
        return validated_data.model_dump() # Return as dict
    except ValidationError as e:
        print(f"[ERROR] Refund policy result validation failed: {e}")
        # In a real application, you might log this error and handle it gracefully
        # For now, we'll return the raw parsed content
        return parsed_content

def send_for_human_approval(data: dict):
    """Simulates sending refund data for human approval."""
    print("\n--- Sending for Human Approval ---")
    print("The following details require human review:")
    print(json.dumps(data, indent=2))
    # In a real system, this would involve sending an email, API call, etc.
def process_refund(email_data: str, Refund_policy: str, evaluation_date: datetime.date = None):
    """Processes a refund after human approval."""
    # Step 1: Extract details from email
    extracted_data = extract_email_details(email_data)
    print("\n--- Extracted Email Details ---")
    print(json.dumps(extracted_data, indent=2))

    # Step 1.1: Get the refund policy rule list
    refund_rules = refund_rules_list(Refund_policy)

    # Step 2: Apply refund policy using LLM, allowing evaluation_date to default to system current date
    policy_result = apply_refund_policy(extracted_data, refund_rules, evaluation_date=evaluation_date)
    print("\n--- Refund Policy Evaluation ---")
    print(json.dumps(policy_result, indent=2))

    # Combine results for a comprehensive view
    final_result = {**extracted_data, **policy_result}
    print("\n--- Combined Result ---")
    print(json.dumps(final_result, indent=2))

    # Step 3: Check for human approval
    if (policy_result['refund_possible'] == "NOT SURE" or extracted_data.get('amount', 0)>50): # Use .get() for safe access
        send_for_human_approval(final_result)
        print("\n--- Refund send for approval---")
    else :
      print("\n--- Processing Refund ---")
# --- Main execution ---
Refund_policy = '1. The date of order should not be older than 3 days. 2. Amount of order can not be more than 50 Rupees.'
base_email_body = "I have an order with id 1232342344 and it should be refunded as item is not working. The amount of order is 30 rupees. The date of order is 20 Sept 2026"

print("\n--- Test Case 1: Order Date: 2026-09-20, Evaluation Date: 2026-09-23 (3 days old) --- ")
process_refund(base_email_body, Refund_policy, evaluation_date=datetime.date(2026, 9, 23))

print("\n--- Test Case 2: Order Date: 2026-09-20, Evaluation Date: 2026-09-24 (4 days old) --- ")
process_refund(base_email_body, Refund_policy, evaluation_date=datetime.date(2026, 9, 24))

print("\n--- Original Test Case: Order Date 25 Sept 2026 (using system current date) ---")
email_body = "I have an order with id 1232342344 and it should be refunded as item is not working. The amount of order is 30 rupees. The date of order is 25 Sept 2026"
process_refund(email_body, Refund_policy)

print("\n--- Original Test Case: Amount > 50, Order Date 20 Sept 2026 (using system current date) ---")
email_body = "I have an order with id A1232342344 and it should be refunded as item is not working. The amount of order is 60 rupees. The date of order is 20 Sept 2026"
process_refund(email_body, Refund_policy)

print("\n--- Original Test Case: Amount > 50, Order Date 20 Sept 2025 (very old, using system current date) ---")
email_body = "I have an order with id A123234234 and it should be refunded as item is not working. The amount of order is 60 rupees. The date of order is 20 Sept 2025"
process_refund(email_body, Refund_policy)


--- Test Case 1: Order Date: 2026-09-20, Evaluation Date: 2026-09-23 (3 days old) --- 
email evaluation cost : CompletionUsage(completion_tokens=199, prompt_tokens=305, total_tokens=504, completion_time=0.415486817, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=163), prompt_time=0.080875749, prompt_tokens_details=None, queue_time=0.185514973, total_time=0.496362566)

--- Extracted Email Details ---
{
  "category": "refund",
  "order_number": "1232342344",
  "amount": 30.0,
  "order_date": "2026-09-20"
}
[
  {
    "rule": "Rule 1",
    "value": "The date of order should not be older than 3 days."
  },
  {
    "rule": "Rule 2",
    "value": "Amount of order can not be more than 50 Rupees."
  }
]
policy evaluation cost : CompletionUsage(completion_tokens=282, prompt_tokens=346, total_tokens=628, completion_time=0.58465936, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=193), prompt_time=0.015614179, prompt_tokens_details=None, queue_time=0.0739810


--- Test Case 1: Order Date: 2026-09-20, Evaluation Date: 2026-09-23 (3 days old) --- 
email evaluation cost : CompletionUsage(completion_tokens=199, prompt_tokens=305, total_tokens=504, completion_time=0.421463046, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=163), prompt_time=0.106305903, prompt_tokens_details=None, queue_time=0.151923574, total_time=0.527768949)

--- Extracted Email Details ---
{
  "category": "refund",
  "order_number": "1232342344",
  "amount": 30.0,
  "order_date": "2026-09-20"
}
[
  {
    "rule": "Rule 1",
    "value": "The date of order should not be older than 3 days."
  },
  {
    "rule": "Rule 2",
    "value": "Amount of order can not be more than 50 Rupees."
  }
]
policy evaluation cost : CompletionUsage(completion_tokens=335, prompt_tokens=346, total_tokens=681, completion_time=0.695217453, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=272), prompt_time=0.047265296, prompt_tokens_details=None, queue_time=0.122036

# Automated Refund Policy Evaluation with LLMs

This notebook demonstrates an end-to-end system for automating refund policy evaluation using a Language Model (LLM) for both data extraction from customer emails and policy application. It includes robust schema validation to ensure the LLM's output adheres to predefined structures.

## 1. Setup and Dependencies

First, install the necessary Python libraries:

```python
!pip install --quiet groq pydantic
```

An API key for the Groq API (or any compatible LLM service) is required. This key should be stored securely, e.g., in Colab's `userdata` secrets manager:

```python
from google.colab import userdata
key = userdata.get('GROQ_API_KEY') # Replace 'GROQ_API_KEY' with your secret name
```

## 2. Pydantic Models for Schema Validation

To ensure the LLM's output conforms to expected JSON schemas, `pydantic` models are defined. These models provide data types, enums, and even regex patterns for fields.

```python
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, Literal

class EmailDetails(BaseModel):
    category: Literal["refund", "shipping", "complaint", "other"]
    order_number: Optional[str]
    amount: float
    order_date: Optional[str] = Field(pattern="^\\d{4}-\\d{2}-\\d{2}$")

class RefundPolicyResult(BaseModel):
    refund_possible: Literal["YES", "NO", "NOT SURE"]
    reason: str
```

## 3. Core Functions

### `refund_rules_list(refund_policy: str)`

This helper function parses a human-readable refund policy string into a list of structured JSON objects, making it easier to present to the LLM.

```python
# Example usage:
# Refund_policy = '1. The date of order should not be older than 3 days. 2. Amount of order can not be more than 50 Rupees.'
# refund_rules = refund_rules_list(Refund_policy)
```

### `extract_email_details(email_body: str, model: str = "openai/gpt-oss-120b")`

This function uses an LLM to extract key factual details (category, order number, amount, order date) from an email body. The extraction is guided by a JSON schema provided in the prompt. After the LLM generates a response, it is validated against the `EmailDetails` Pydantic model.

### `apply_refund_policy(extracted_data: dict, refund_policy: list, model: str = "openai/gpt-oss-120b", evaluation_date: datetime.date = None)`

This function applies the defined refund policy to the `extracted_data`. Instead of hardcoding the policy logic in Python, it leverages the LLM to interpret the `refund_policy` rules (passed as a list of JSON objects) and the `extracted_data`, along with an `evaluation_date`. The LLM returns a `refund_possible` status (`YES`, `NO`, `NOT SURE`) and a `reason`. The output is then validated against the `RefundPolicyResult` Pydantic model.

If `evaluation_date` is not provided, it defaults to the system's current date.

### `send_for_human_approval(data: dict)`

A placeholder function that simulates sending cases requiring human intervention to a review queue. This is called when the `refund_possible` status is `NOT SURE` or if specific business rules (e.g., amount > 50) trigger manual review.

### `process_refund(email_data: str, Refund_policy: str, evaluation_date: datetime.date = None)`

This orchestrator function ties everything together:
1.  Calls `extract_email_details` to get structured data from the email.
2.  Parses the `Refund_policy` string into a structured list.
3.  Calls `apply_refund_policy` to get the LLM's refund decision.
4.  Combines the extracted data and policy result into a `final_result`.
5.  Checks the `refund_possible` status and decides whether to `send_for_human_approval` or proceed with `Processing Refund`.

## 4. Usage Examples and Testing

The notebook includes several test cases to demonstrate the `process_refund` function's behavior under different scenarios, including explicit date boundary tests for the 3-day policy rule and cases involving amounts exceeding the threshold.

```python
# --- Main execution ---
Refund_policy = '1. The date of order should not be older than 3 days. 2. Amount of order can not be more than 50 Rupees.'
base_email_body = "I have an order with id 1232342344 and it should be refunded as item is not working. The amount of order is 30 rupees. The date of order is 20 Sept 2026"

print("\n--- Test Case 1: Order Date: 2026-09-20, Evaluation Date: 2026-09-23 (3 days old) --- ")
process_refund(base_email_body, Refund_policy, evaluation_date=datetime.date(2026, 9, 23))

print("\n--- Test Case 2: Order Date: 2026-09-20, Evaluation Date: 2026-09-24 (4 days old) --- ")
process_refund(base_email_body, Refund_policy, evaluation_date=datetime.date(2026, 9, 24))

# ... (other test cases)
```

## 5. Key Takeaways

*   **LLM for Structured Data Extraction**: Demonstrates effective use of LLMs for extracting specific data points from unstructured text (emails) into a machine-readable JSON format.
*   **LLM for Policy Application**: Shows how LLMs can be used to interpret and apply complex business rules (refund policies) by providing them with the rules and relevant data in a clear prompt.
*   **Schema Conformance with Pydantic**: Emphasizes the importance of validating LLM output against a predefined schema to ensure data quality and reliability in downstream processes, going beyond basic JSON syntax validation.
*   **Human-in-the-Loop**: Integrates a mechanism for human approval for ambiguous or high-risk cases, ensuring a balanced automated system.

This approach provides a flexible and scalable way to handle customer service automation, where policies or data structures can be easily updated by modifying prompts and Pydantic models rather than extensive code changes.